In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Redes Neurais Artificiais
## I. O Neurônio Artificial e o Perceptron

### 1.1 Introdução Teórica

O ponto de partida das Redes Neurais é o Neurônio Artificial, inspirado na célula nervosa biológica. Ele realiza duas operações principais:

1. Soma Ponderada: Multiplica cada entrada ($x_i$) por seu respectivo peso ($w_i$) e soma um bias ($b$).
$$z=\left(\sum_{i=1}^{n} x_i w_i \right)+b$$

2. Ativação: Aplica uma função de ativação ($f$) ao resultado ($z$) para produzir a saída:
$$r=f(z)$$

O Perceptron (1957) é o modelo mais simples, usando uma função de ativação Degrau para classificação binária. Ele só consegue resolver problemas linearmente separáveis.

### 1.2 Implementação do Perceptron (Classificação Linear)

Vamos simular a porta lógica `AND` para demonstrar o Perceptron.

In [ ]:
# Definindo a porta lógica AND
data = {
    'X1': [0, 0, 1, 1],
    'X2': [0, 1, 0, 1],
    'Y': [0, 0, 0, 1]
}
df_and = pd.DataFrame(data)
X = df_and[['X1', 'X2']].values
Y = df_and['Y'].values

In [ ]:
# Função de Ativação Degrau
def step_function(z):
    return 1 if z >= 0 else 0

# Treinamento do Perceptron
def train_perceptron(X, Y, epochs=10, learning_rate=0.1):
    # Inicializa pesos (w1, w2) e bias (b) aleatoriamente (ou zeros)
    weights = np.zeros(X.shape[1])  # 2 pesos (w1, w2)
    bias = 0.0

    for epoch in range(epochs):
        total_error = 0
        for x, y_true in zip(X, Y):
            # 1. Soma Ponderada
            z = np.dot(x, weights) + bias

            # 2. Ativação
            y_pred = step_function(z)

            # 3. Cálculo do Erro e Ajuste
            error = y_true - y_pred
            total_error += abs(error)

            # 4. Atualização dos Pesos e Bias
            weights += learning_rate * error * x
            bias += learning_rate * error

        print(f"Época {epoch+1}: Erro Total = {total_error}")
        if total_error == 0:
            print("Convergência atingida!")
            break

    return weights, bias

In [ ]:
# Execução do treinamento para AND
print("--- Treinando Perceptron (Porta AND) ---")
final_weights, final_bias = train_perceptron(X, Y, epochs=10, learning_rate=0.1)

print(f"\nPesos Finais (w1, w2): {final_weights}")
print(f"Bias Final (b): {final_bias}")

# Teste
def predict_perceptron(x, weights, bias):
    z = np.dot(x, weights) + bias
    return step_function(z)

print("\n--- Teste de Previsão ---")
for x, y_true in zip(X, Y):
    y_pred = predict_perceptron(x, final_weights, final_bias)
    print(f"Entrada: {x}, Verdadeiro: {y_true}, Previsto: {y_pred}")

# II. MLP

## 2.1 Teoria da MLP

O Perceptron falha no problema `XOR` porque ele não é linearmente separável. A solução é o MLP (Multi-Layer Perceptron), que adiciona Camadas Ocultas.

Para que camadas ocultas sejam eficazes, a função de ativação deve ser Não-Linear e, mais importante para o próximo passo, Diferenciável.

1. **Função Sigmoide**: Usada historicamente.
$$f(z)=\frac{1}{1 + e^{−z}}$$

2. **Função ReLU** (Rectified Linear Unit): A mais comum atualmente, por ser simples e eficiente.
$$f(z)=\max(0,z)$$

## 2.2 Preparando o Problema XOR

In [ ]:
# Definindo o problema XOR
xor_data = {
    'X1': [0, 0, 1, 1],
    'X2': [0, 1, 0, 1],
    'Y': [0, 1, 1, 0]
}
df_xor = pd.DataFrame(xor_data)
X_xor = df_xor[['X1', 'X2']].values
Y_xor = df_xor['Y'].values.reshape(-1, 1)

In [ ]:
# Funções de Ativação e Suas Derivadas
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    return a * (1 - a)


#### Estrutura da MLP:
* Camada de Entrada: 2 neurônios (X1, X2)
* Camada Oculta: 3 neurônios (escolha arbitrária)
* Camada de Saída: 1 neurônio (Y)

# III. Backpropagation

## 3.1 Teoria do Treinamento

O **Backpropagation** (**Retropropagação**) é o algoritmo para treinar MLPs. Ele usa a **Descida do Gradiente** para ajustar os pesos de trás para frente, minimizando a Função de Custo (usaremos o **Erro Quadrático Médio** `MSE`, $J$).

A atualização de um peso ($w$) é dada por:
$$w_{i}=w_{i-1} - \alpha \frac{\partial J}{\partial w}$$

O cálculo do gradiente ($\frac{\partial J}{\partial w}$) é feito pela **Regra da Cadeia**, propagando o erro da camada de saída de volta para as camadas ocultas.

## 3.2 Implementação da MLP com Backpropagation

In [ ]:
np.random.seed(42)

# Hiperparâmetros
epochs = 10000
learning_rate = 0.5
input_size = 2
hidden_size = 3
output_size = 1

# 1. Inicialização dos Pesos e Bias
# Camada de Entrada -> Oculta (W1 e b1)
W1 = np.random.uniform(size=(input_size, hidden_size))
b1 = np.zeros((1, hidden_size))

# Camada Oculta -> Saída (W2 e b2)
W2 = np.random.uniform(size=(hidden_size, output_size))
b2 = np.zeros((1, output_size))

# 2. Loop de Treinamento
for epoch in range(epochs):

    # --- A. FORWARD PASS (Propagação Direta) ---
    # Camada Oculta
    Z1 = np.dot(X_xor, W1) + b1
    R1 = sigmoid(Z1)

    # Camada de Saída
    Z2 = np.dot(R1, W2) + b2
    R2 = sigmoid(Z2)

    # --- B. BACKWARD PASS (Retropropagação do Erro) ---
    # 1. Erro na Camada de Saída
    # dJ/dA2 (Derivada do Custo em relação à Saída)
    E_output = Y_xor - R2

    # dJ/dZ2: Gradiente de Z2
    dZ2 = E_output * sigmoid_derivative(R2)

    # 2. Erro na Camada Oculta
    E_hidden = np.dot(dZ2, W2.T)

    # dJ/dZ1: Gradiente de Z1
    dZ1 = E_hidden * sigmoid_derivative(R1)

    # --- C. ATUALIZAÇÃO DOS PESOS (Gradient Descent) ---
    # Gradientes
    dW2 = np.dot(R1.T, dZ2)
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dW1 = np.dot(X_xor.T, dZ1)
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    # Ajuste
    W2 += learning_rate * dW2
    b2 += learning_rate * db2

    W1 += learning_rate * dW1
    b1 += learning_rate * db1

    # Exibir Erro
    if epoch % 1000 == 0:
        loss = np.mean(np.square(E_output)) / 2
        print(f"Época {epoch}: Custo (Loss) = {loss:.4f}")

print("\n--- Treinamento Concluído ---")

## 3.3 Teste e Avaliação

In [ ]:
# Função de Previsão Final
def predict_mlp(X, W1, b1, W2, b2):
    # Forward Pass
    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)
    # Converte para binário (0 ou 1)
    return (A2 > 0.5).astype(int)

# Previsão
Y_pred = predict_mlp(X_xor, W1, b1, W2, b2)

print("--- Resultado Final (Problema XOR) ---")
print("Entrada (X1, X2) | Verdadeiro (Y) | Previsto (Y_pred)")
print("-----------------|----------------|----------------")
for i in range(len(X_xor)):
    print(f"({X_xor[i][0]}, {X_xor[i][1]})      | {Y_xor[i][0]}              | {Y_pred[i][0]}")

# Acurácia
accuracy = np.mean(Y_pred == Y_xor)
print(f"\nAcurácia: {accuracy * 100:.2f}%")

# IV. Framework Moderno: TensorFlow/Keras

## 4.1 Teoria da Abstração

No TensorFlow com a API **Keras**, o processo se resume a três etapas simples:

1. **Definir a Arquitetura**: Criar as camadas (Entrada, Oculta, Saída), especificando o número de neurônios e a **função de ativação** para cada uma.

2. **Compilar o Modelo**: Escolher o otimizador (usaremos o "Adam", uma versão mais avançada da Descida do Gradiente) e a **função de perda** (loss).

3. **Treinar o Modelo**: Passar os dados de entrada ($X$) e as saídas desejadas ($Y$), definindo o número de épocas (iterações).

## 4.2 Implementação do Problema XOR com TensorFlow/Keras

Vamos recriar a mesma arquitetura: 2 neurônios (entrada) $→$ 3 neurônios (oculta) $→$ 1 neurônio (saída), utilizando a ativação Sigmoide.

In [ ]:
import tensorflow as tf
from tensorflow import keras

sns.set_theme()

In [ ]:
# --- 2. Definição da Arquitetura (Sequential Model) ---
model = keras.Sequential([
    keras.layers.Dense(units=3, activation='sigmoid', input_shape=(2,)),

    keras.layers.Dense(units=1, activation='sigmoid')
])

In [ ]:

# Resumo do modelo
model.summary()


In [ ]:
# --- 3. Compilação do Modelo ---
model.compile(optimizer='adam',
              loss='mse',
              metrics=['accuracy'])

In [ ]:
# --- 4. Treinamento do Modelo ---
history = model.fit(X_xor, Y_xor,
                    epochs=1000,
                    verbose=0)

In [ ]:
# --- 5. Avaliação e Visualização ---

# Previsões Finais
print("\n--- Teste de Previsão (Keras) ---")
predictions_raw = model.predict(X_xor)
predictions_binary = (predictions_raw > 0.5).astype(int)

print("Entrada (X1, X2) | Verdadeiro (Y) | Previsto (Y_pred)")
print("-----------------|----------------|----------------")
for i in range(len(X_xor)):
    print(f"({X_xor[i][0]}, {X_xor[i][1]})      | {Y_xor[i][0]}              | {predictions_binary[i][0]}")


In [ ]:
# Acurácia
loss, accuracy = model.evaluate(X_xor, Y_xor, verbose=0)
print(f"\nAcurácia Keras: {accuracy * 100:.2f}%")

In [ ]:
# --- 6. Visualização ---

# Extraindo o histórico da perda (loss) ao longo das épocas
loss_values = history.history['loss']

plt.figure(figsize=(10, 6))
plt.plot(loss_values, label='Função de Perda (Loss)')
plt.title('Evolução da Função de Perda Durante o Treinamento')
plt.xlabel('Época')
plt.ylabel('Loss (Erro Quadrático Médio)')
plt.grid(True)
plt.legend()
plt.show()

# V. Representação visual

In [ ]:
x_min, x_max = -0.5, 1.5
y_min, y_max = -0.5, 1.5

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))

grid_points = np.c_[xx.ravel(), yy.ravel()]

Z_raw = model.predict(grid_points, verbose=0)

Z = Z_raw.reshape(xx.shape)

plt.figure(figsize=(10, 8))

plt.contourf(xx, yy, Z, levels=1, cmap=plt.cm.RdBu, alpha=0.6)

plt.contour(xx, yy, Z, levels=[0.5], linewidths=2, colors='k')

sns.scatterplot(x=X_xor[:, 0], y=X_xor[:, 1], hue=Y_xor.flatten(),
                palette={0: 'blue', 1: 'red'}, style=Y_xor.flatten(),
                markers={0: 'o', 1: 'X'}, s=200, edgecolor='black',
                label='Dados XOR')

plt.title('Fronteira de Decisão Não-Linear (MLP Treinado para XOR)')
plt.xlabel('X1')
plt.ylabel('X2')
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.legend(title='Classe', loc='upper right')
plt.grid(True)
plt.show()